# ДЗ 3. Аналіз даних у PySpark

## Імпорти та Spark-сесія

In [1]:
import gdown
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("goit-de-hw-03").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/21 19:42:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Завантаження CSV-файлів у DataFrame

In [2]:
FILE_IDS = {
    "users": "1GN8o8vYP8LLj3A0UWkvbgnf0YjcF3PkV",
    "purchases": "1cDLTc0KYQfnsDWHLR0cJkF5puBpAo2Uh",
    "products": "1im5cGVl1z8ejypTyNJZSQ2lqMGh6tTFN",
}


def load_csv(name, file_id):
    """Завантажує CSV з Google Drive та читає його як DataFrame."""
    path = gdown.download(id=file_id, output=f"{name}.csv", quiet=True)
    return spark.read.csv(path, header=True, inferSchema=True)


raw = {name: load_csv(name, file_id) for name, file_id in FILE_IDS.items()}

for name, df in raw.items():
    print(f"{name}: {df.count()} рядків")
    df.show(5)

users: 100 рядків
+-------+------+---+-----------------+
|user_id|  name|age|            email|
+-------+------+---+-----------------+
|      1|User_1| 45|user1@example.com|
|      2|User_2| 48|user2@example.com|
|      3|User_3| 36|user3@example.com|
|      4|User_4| 46|user4@example.com|
|      5|User_5| 29|user5@example.com|
+-------+------+---+-----------------+
only showing top 5 rows
purchases: 200 рядків
+-----------+-------+----------+----------+--------+
|purchase_id|user_id|product_id|      date|quantity|
+-----------+-------+----------+----------+--------+
|          1|     52|         9|2022-01-01|       1|
|          2|     93|        37|2022-01-02|       8|
|          3|     15|        33|2022-01-03|       1|
|          4|     72|        42|2022-01-04|       9|
|          5|     61|        44|2022-01-05|       6|
+-----------+-------+----------+----------+--------+
only showing top 5 rows


products: 50 рядків
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|         1|   Product_1|     Beauty|  8.3|
|         2|   Product_2|       Home|  8.3|
|         3|   Product_3|Electronics|  9.2|
|         4|   Product_4|Electronics|  2.6|
|         5|   Product_5|Electronics|  9.4|
+----------+------------+-----------+-----+
only showing top 5 rows


## 2. Очищення даних від пропущених значень

In [3]:
clean = {name: df.dropna() for name, df in raw.items()}

for name, df in clean.items():
    print(f"{name}: {raw[name].count()} -> {df.count()} рядків")
    df.show(5)

users, purchases, products = clean["users"], clean["purchases"], clean["products"]

users: 100 -> 95 рядків


+-------+------+---+-----------------+
|user_id|  name|age|            email|
+-------+------+---+-----------------+
|      1|User_1| 45|user1@example.com|
|      2|User_2| 48|user2@example.com|
|      3|User_3| 36|user3@example.com|
|      4|User_4| 46|user4@example.com|
|      5|User_5| 29|user5@example.com|
+-------+------+---+-----------------+
only showing top 5 rows
purchases: 200 -> 195 рядків
+-----------+-------+----------+----------+--------+
|purchase_id|user_id|product_id|      date|quantity|
+-----------+-------+----------+----------+--------+
|          1|     52|         9|2022-01-01|       1|
|          2|     93|        37|2022-01-02|       8|
|          3|     15|        33|2022-01-03|       1|
|          4|     72|        42|2022-01-04|       9|
|          5|     61|        44|2022-01-05|       6|
+-----------+-------+----------+----------+--------+
only showing top 5 rows


products: 50 -> 47 рядків
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|         1|   Product_1|     Beauty|  8.3|
|         2|   Product_2|       Home|  8.3|
|         3|   Product_3|Electronics|  9.2|
|         4|   Product_4|Electronics|  2.6|
|         5|   Product_5|Electronics|  9.4|
+----------+------------+-----------+-----+
only showing top 5 rows


## 3. Загальна сума покупок за категоріями

In [4]:
def spending_by_category(df):
    """Повертає суму покупок за кожною категорією продуктів."""
    return (
        df.groupBy("category")
        .agg(F.round(F.sum("total"), 2).alias("total_spent"))
        .orderBy(F.desc("total_spent"))
    )


purchases_full = purchases.join(products, "product_id").withColumn(
    "total", F.col("quantity") * F.col("price")
)

spending_by_category(purchases_full).show()

+-----------+-----------+
|   category|total_spent|
+-----------+-----------+
|     Sports|     1802.5|
|       Home|     1523.5|
|Electronics|     1174.8|
|   Clothing|      790.3|
|     Beauty|      459.9|
+-----------+-----------+



## 4. Сума покупок за категоріями для віку 18–25 включно

In [5]:
young_purchases = purchases_full.join(users, "user_id").filter(
    F.col("age").between(18, 25)
)

young_by_category = spending_by_category(young_purchases)
young_by_category.show()

+-----------+-----------+
|   category|total_spent|
+-----------+-----------+
|       Home|      361.1|
|     Sports|      310.5|
|Electronics|      249.6|
|   Clothing|      245.0|
|     Beauty|       41.4|
+-----------+-----------+



## 5. Частка покупок за категоріями від сумарних витрат віку 18–25

In [6]:
young_total = young_purchases.agg(F.sum("total")).first()[0]

young_share = young_by_category.withColumn(
    "percent", F.round(F.col("total_spent") / young_total * 100, 2)
)

print(f"Сумарні витрати віку 18–25: {young_total:.2f}")
young_share.show()

Сумарні витрати віку 18–25: 1207.60


+-----------+-----------+-------+
|   category|total_spent|percent|
+-----------+-----------+-------+
|       Home|      361.1|   29.9|
|     Sports|      310.5|  25.71|
|Electronics|      249.6|  20.67|
|   Clothing|      245.0|  20.29|
|     Beauty|       41.4|   3.43|
+-----------+-----------+-------+



## 6. Топ-3 категорії за відсотком витрат віку 18–25

In [7]:
top_categories = young_share.orderBy(F.desc("percent")).limit(3)
top_categories.show()

+-----------+-----------+-------+
|   category|total_spent|percent|
+-----------+-----------+-------+
|       Home|      361.1|   29.9|
|     Sports|      310.5|  25.71|
|Electronics|      249.6|  20.67|
+-----------+-----------+-------+



## Висновки

In [8]:
print("Найбільша частка витрат покупців віком 18–25 років:")
for row in top_categories.collect():
    print(f"  {row['category']}: {row['percent']:.2f}% ({row['total_spent']:.2f})")

Найбільша частка витрат покупців віком 18–25 років:


  Home: 29.90% (361.10)
  Sports: 25.71% (310.50)
  Electronics: 20.67% (249.60)
